In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1eeAEwx5BNVDUgOULg1nC-ckX2cNMmv8f", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Agentic Loop Fundamentals — Hands-On Lab

Welcome to your first hands-on lab for Domain 1 of the Claude Certified Architect exam. In this notebook, we will **build a working agentic loop from scratch** — the same while-loop pattern that powers every Claude Code session and every Agent SDK deployment.

By the end of this notebook, you will be able to:
- Implement the core agentic loop with `stop_reason`-driven control flow
- Trace multi-turn tool execution through conversation history
- Identify and fix the two critical anti-patterns the exam tests
- Build a mock agent that resolves customer support queries autonomously

## AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/agentic-architecture/practice/1/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you are building a customer support agent. A customer asks: "What is the status of order ORD-1234?" Your agent needs to look up the order, check shipping status, and respond — all without you writing a single if/else branch to decide which tool to call.

The **agentic loop** is what makes this possible. It is the fundamental control structure behind every agent: a while loop where the model itself decides what to do next. Understanding this loop — and the `stop_reason` field that drives it — is the foundation for the entire 27% of your exam that covers agentic architecture.

Let us build it from scratch.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — What Makes an Agent "Agentic"?

In traditional programming, the developer controls the flow. In an agentic system, the **model** controls the flow — it decides which tools to call, in what order, and when the task is done.

In [ ]:
# Setup
import json
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

# Traditional approach
def handle_query_traditional(query):
    order_id = "ORD-1234"
    status = {"status": "shipped"}
    return f"Order {order_id}: {status}"

# Agentic approach — model decides everything
# while True:
#     response = call_model(messages)
#     if response.stop_reason == "end_turn": break
#     if response.stop_reason == "tool_use":
#         execute_tool_and_append_result()

print("Core insight: the MODEL drives the loop, not the developer.")
print("The stop_reason field is the steering wheel.")
print()
print("Traditional:", handle_query_traditional("order status"))

In [ ]:
#@title 🎧 Listen: Mock Api
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_mock_api.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Core Implementation — The Mock API

In [ ]:
@dataclass
class ToolUseBlock:
    type: str = "tool_use"
    id: str = field(default_factory=lambda: f"toolu_{uuid.uuid4().hex[:12]}")
    name: str = ""
    input: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TextBlock:
    type: str = "text"
    text: str = ""

@dataclass
class MockResponse:
    content: list = field(default_factory=list)
    stop_reason: str = "end_turn"
    model: str = "claude-sonnet-4-20250514"

ORDER_DB = {
    "ORD-1234": {"status": "shipped", "tracking": "TRK-5678", "item": "Wireless Headphones", "amount": 79.99},
    "ORD-5555": {"status": "processing", "tracking": None, "item": "USB-C Cable", "amount": 12.99},
    "ORD-9999": {"status": "delivered", "tracking": "TRK-1111", "item": "Laptop Stand", "amount": 45.00},
}

SHIPPING_DB = {
    "TRK-5678": {"carrier": "FedEx", "eta": "2024-03-15", "last_location": "Memphis, TN"},
    "TRK-1111": {"carrier": "UPS", "eta": "2024-03-10", "last_location": "Delivered"},
}

def execute_tool(tool_name, tool_input):
    if tool_name == "lookup_order":
        oid = tool_input.get("order_id", "")
        return json.dumps(ORDER_DB.get(oid, {"error": f"Not found: {oid}"}))
    elif tool_name == "check_shipping":
        tid = tool_input.get("tracking_id", "")
        return json.dumps(SHIPPING_DB.get(tid, {"error": f"Not found: {tid}"}))
    elif tool_name == "process_refund":
        return json.dumps({"success": True, "refund_id": f"REF-{uuid.uuid4().hex[:6]}", "amount": tool_input.get("amount", 0)})
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

print("lookup_order:", execute_tool("lookup_order", {"order_id": "ORD-1234"}))
print("check_shipping:", execute_tool("check_shipping", {"tracking_id": "TRK-5678"}))

In [ ]:
class MockClaudeClient:
    def __init__(self):
        self.call_count = 0
        self.scenario = []

    def set_scenario(self, responses):
        self.scenario = responses
        self.call_count = 0

    def create_message(self, model, max_tokens, tools, messages):
        if self.call_count < len(self.scenario):
            r = self.scenario[self.call_count]
            self.call_count += 1
            return r
        return MockResponse(content=[TextBlock(text="Done.")], stop_reason="end_turn")

TOOLS = [
    {"name": "lookup_order", "description": "Look up order by ID",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}},
    {"name": "check_shipping", "description": "Check shipping by tracking ID",
     "input_schema": {"type": "object", "properties": {"tracking_id": {"type": "string"}}, "required": ["tracking_id"]}},
    {"name": "process_refund", "description": "Process a refund",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}}, "required": ["order_id", "amount"]}},
]

client = MockClaudeClient()
print(f"Mock client ready with {len(TOOLS)} tools: {[t['name'] for t in TOOLS]}")

In [ ]:
#@title 🎧 Listen: Agentic Loop
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_agentic_loop.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. The Agentic Loop — The Heart of Every Agent

In [ ]:
def run_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"

print("Agentic loop defined!")

In [ ]:
#@title 🎧 Listen: Tracing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_tracing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Tracing a Multi-Turn Execution

In [ ]:
client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="check_shipping", input={"tracking_id": "TRK-5678"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Your order ORD-1234 (Headphones, $79.99) shipped via FedEx. In Memphis, TN. Arriving March 15. Tracking: TRK-5678.")], stop_reason="end_turn"),
])
result = run_agentic_loop(client, TOOLS, "What is the status of order ORD-1234?")

**Turn 1:** Model called `lookup_order`. **Turn 2:** Called `check_shipping`. **Turn 3:** Had all info, responded with `end_turn`. The model made every decision. This is exactly what we want.

In [ ]:
#@title 🎧 Listen: Anti Patterns
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_anti_patterns.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Anti-Pattern Detection Lab

### Anti-Pattern 1: Natural Language Parsing

In [ ]:
def broken_loop_v1(client, tools, user_message, max_turns=10):
    messages = [{"role": "user", "content": user_message}]
    for turn in range(max_turns):
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)
        for block in response.content:
            if hasattr(block, "text") and "done" in block.text.lower():
                return f"[EARLY EXIT] {block.text}"
        if response.stop_reason == "tool_use":
            for block in response.content:
                if hasattr(block, "name"):
                    result = execute_tool(block.name, block.input)
                    messages.append({"role": "assistant", "content": response.content})
                    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": block.id, "content": result}]})
                    break
    return "Max turns"

client.set_scenario([
    MockResponse(content=[TextBlock(text="Done checking DB, now shipping..."), ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Shipped via FedEx.")], stop_reason="end_turn"),
])
print("Anti-Pattern 1: NL parsing")
print(f"Result: {broken_loop_v1(client, TOOLS, 'Check ORD-1234')}")
print("Bug: 'done checking' triggered exit mid-task!")

In [ ]:
#@title 🎧 Listen: Anti Pattern 2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_anti_pattern_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Anti-Pattern 2: Iteration Caps

In [ ]:
def broken_loop_v2(client, tools, msg):
    messages = [{"role": "user", "content": msg}]
    for i in range(3):
        r = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)
        if r.stop_reason == "tool_use":
            for b in r.content:
                if hasattr(b, "name"):
                    res = execute_tool(b.name, b.input)
                    messages.append({"role": "assistant", "content": r.content})
                    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": b.id, "content": res}]})
                    break
    return "Completed 3 iterations"

client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-5555"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="ORD-5555 processing.")], stop_reason="end_turn"),
    MockResponse(content=[TextBlock(text="Already answered.")], stop_reason="end_turn"),
])
print("Anti-Pattern 2: Iteration cap")
print(f"Result: {broken_loop_v2(client, TOOLS, 'Status ORD-5555?')}")
print("Bug: Model done in turn 2, loop forced turn 3!")

### Comparison

In [ ]:
print(f"{'Scenario':<25} {'Correct':<10} {'Cap=3':<10} {'Issue'}")
print("=" * 58)
for n, t in [("2-turn", 2), ("1-turn", 1), ("3-turn", 3), ("5-turn", 5)]:
    w = max(0, 3 - t)
    m = max(0, t - 3)
    p = f"{w} wasted" if w else (f"{m} MISSED" if m else "OK")
    print(f"  {n:<23} {t:<10} {3:<10} {p}")

In [ ]:
#@title 🎧 Listen: Todo Exercises
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_todo_exercises.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Your Turn — TODO Exercises

### Exercise 1: Refund Agent

In [ ]:
# ============ TODO ============
# 3-turn scenario: lookup ORD-9999, process_refund($45), confirm
client.set_scenario([
    # YOUR CODE HERE
])
# result = run_agentic_loop(client, TOOLS, "Refund ORD-9999")
# ==============================

In [ ]:
#@title 🎧 Listen: Todo Fix Loop
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_todo_fix_loop.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Exercise 2: Fix the Broken Loop

In [ ]:
# ============ TODO ============
def fixed_loop(client, tools, msg, max_turns=10):
    messages = [{"role": "user", "content": msg}]
    # YOUR CODE HERE
    pass
# ==============================

In [ ]:
#@title 🎧 Listen: Todo Logger
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_todo_logger.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Exercise 3: Conversation Logger

In [ ]:
# ============ TODO ============
# Return (final_text, messages_list)
# YOUR CODE HERE
# ==============================

In [ ]:
#@title 🎧 Listen: Putting Together
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_putting_together.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Putting It All Together

In [ ]:
client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="check_shipping", input={"tracking_id": "TRK-5678"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-5555"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Both orders:\n1. ORD-1234: Shipped, FedEx, March 15\n2. ORD-5555: Processing, 1-2 days")], stop_reason="end_turn"),
])
run_agentic_loop(client, TOOLS, "Check orders ORD-1234 and ORD-5555")

In [ ]:
#@title 🎧 Listen: Exam Practice
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_exam_practice.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Exam Practice Questions

In [ ]:
for i, (q, a, w) in enumerate([
    ("stop_reason='tool_use' but text says 'we are done.' Action?", "Execute tool, continue loop", "stop_reason is authoritative."),
    ("for i in range(5): call_model() — what's wrong?", "Cap is primary control, not stop_reason", "Model drives via end_turn."),
    ("After tool exec, what's appended?", "Assistant response + user tool_result", "Both required by API."),
    ("Two stop_reason values for agentic loop?", "tool_use and end_turn", "tool_use=need tool, end_turn=done."),
], 1):
    print(f"Q{i}: {q}\n   {a} — {w}\n")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 10. Reflection

**Key exam concepts (Task Statement 1.1):**
- Agentic loop checks `stop_reason` after every API call
- `tool_use` → execute, append, loop. `end_turn` → done, exit.
- Never parse NL for termination. Never use iteration caps as primary control.
- Append BOTH assistant response AND tool result.

In [ ]:
print("Notebook complete! Next: Multi-Agent Coordinator Patterns (NB 2)")